# F12-UC3 — YOLOv8 Urban Tree Detection: RGB vs RGB+NIR

**Project:** Micro-Forest Health Monitoring — NEUSTA France  
**Author:** Sofya Tadevosyan  
**Date:** April 2026  

This notebook trains and evaluates two YOLOv8 models:
- **Baseline:** YOLOv8s on standard RGB (3-channel) aerial images
- **Improvement:** YOLOv8s patched to accept RGB+NIR (4-channel) — exploiting the Near-Infrared band for vegetation health

**Runtime required:** GPU (T4 or better). Go to `Runtime → Change runtime type → T4 GPU`.

**Dataset:** NAIP urban tree detection dataset (Ventura et al., 2024) — 1,651 images, 96,547 annotated trees.

---
## Setup checklist before running
1. Runtime → Change runtime type → **T4 GPU**
2. Upload the dataset folder to Google Drive at `My Drive/NEUSTA/Dataset/urban-tree-detection-data-main/`
3. Runtime → Run all

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

## Cell 2 — Install dependencies

In [ ]:
!pip install -q ultralytics rasterio opencv-python-headless tqdm PyYAML pandas matplotlib

## Cell 3 — Mount Google Drive and set paths

**Before running:** upload the dataset folder to Google Drive at:  
`My Drive/NEUSTA/Dataset/urban-tree-detection-data-main/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATASET_DIR = '/content/drive/MyDrive/NEUSTA/Dataset/urban-tree-detection-data-main'
WORK_DIR    = '/content/micro_forest'
YOLO_DIR    = os.path.join(WORK_DIR, 'yolo_dataset')
RESULTS_DIR = os.path.join(WORK_DIR, 'results')

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.isdir(DATASET_DIR), f'Dataset not found at {DATASET_DIR}'
n_tifs = len([f for f in os.listdir(os.path.join(DATASET_DIR, 'images')) if f.endswith('.tif')])
print(f'Dataset found. TIF images: {n_tifs} (expected 1651)')

## Cell 4 — Clone the GitHub repo

In [ ]:
import os
REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SofiaTadevosyan/Micro-Forest-Neusta-Masters-Internship.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

SCRIPTS_DIR = os.path.join(REPO_DIR, 'yolov8_urban_trees')
print('Scripts available:', os.listdir(SCRIPTS_DIR))

## Cell 5 — Convert dataset to YOLO format

Reads all 1,651 TIFF files, exports RGB PNGs + RGBN .npy files, converts point annotations to YOLO boxes (radius=15 px).  
**Takes ~5-10 minutes. Skipped automatically if already done.**

In [ ]:
import os
if not os.path.exists(os.path.join(YOLO_DIR, 'dataset_rgb.yaml')):
    !python3 {SCRIPTS_DIR}/convert_annotations.py \
        --dataset_dir "{DATASET_DIR}" \
        --output_dir  "{YOLO_DIR}" \
        --radius 15
else:
    print('Dataset already converted. Skipping.')

for split in ['train', 'val', 'test']:
    n_rgb  = len(os.listdir(os.path.join(YOLO_DIR, 'images', 'rgb',  split)))
    n_rgbn = len(os.listdir(os.path.join(YOLO_DIR, 'images', 'rgbn', split)))
    n_lbl  = len(os.listdir(os.path.join(YOLO_DIR, 'labels', split)))
    print(f'{split:5s} → RGB: {n_rgb:4d} PNGs | RGBN: {n_rgbn:4d} NPYs | Labels: {n_lbl:4d} TXTs')

## Cell 6 — Train RGB Baseline (YOLOv8s, 3-channel)

Standard YOLOv8s trained on RGB aerial images.  
**Expected time on T4 GPU: ~45-60 minutes for 50 epochs.**

In [ ]:
import os
RGB_YAML    = os.path.join(YOLO_DIR, 'dataset_rgb.yaml')
RGB_WEIGHTS = os.path.join(WORK_DIR, 'runs/train/yolov8_rgb/weights/best.pt')

if not os.path.exists(RGB_WEIGHTS):
    !python3 {SCRIPTS_DIR}/train_rgb.py \
        --data    "{RGB_YAML}" \
        --model   yolov8s.pt \
        --epochs  50 \
        --imgsz   256 \
        --batch   32 \
        --name    yolov8_rgb \
        --project "{WORK_DIR}/runs/train" \
        --device  0
else:
    print('RGB model already trained. Skipping.')

print('RGB weights exist:', os.path.exists(RGB_WEIGHTS))

## Cell 7 — Train RGB+NIR Model (YOLOv8s, 4-channel)

YOLOv8s with first Conv2d patched to accept 4 channels (RGB + Near-Infrared).  
NIR filter initialised as mean of pretrained RGB filters.  
**Expected time on T4 GPU: ~60-75 minutes for 50 epochs.**

In [ ]:
import os
RGBN_YAML    = os.path.join(YOLO_DIR, 'dataset_rgbn.yaml')
RGBN_WEIGHTS = os.path.join(WORK_DIR, 'runs/train/yolov8_rgbn/weights/best.pt')

if not os.path.exists(RGBN_WEIGHTS):
    !python3 {SCRIPTS_DIR}/train_rgbn.py \
        --data    "{RGBN_YAML}" \
        --model   yolov8s.pt \
        --epochs  50 \
        --batch   16 \
        --name    yolov8_rgbn \
        --project "{WORK_DIR}/runs/train" \
        --device  cuda \
        --workers 2
else:
    print('RGBN model already trained. Skipping.')

print('RGBN weights exist:', os.path.exists(RGBN_WEIGHTS))

## Cell 8 — Evaluate and Compare Both Models

In [ ]:
import os
!python3 {SCRIPTS_DIR}/evaluate.py \
    --rgb_weights  "{RGB_WEIGHTS}" \
    --rgbn_weights "{RGBN_WEIGHTS}" \
    --data_rgb     "{RGB_YAML}" \
    --data_rgbn    "{RGBN_YAML}" \
    --output_dir   "{RESULTS_DIR}" \
    --visualise \
    --n_samples 8

## Cell 9 — Display results

In [ ]:
import json, os
import pandas as pd
from IPython.display import Image, display

with open(os.path.join(RESULTS_DIR, 'metrics.json')) as f:
    metrics = json.load(f)

df = pd.DataFrame(metrics).T
df.index = ['RGB (baseline)', 'RGB+NIR']
print('\n=== RESULTS ===')
print(df.to_string(float_format='{:.4f}'.format))

display(Image(os.path.join(RESULTS_DIR, 'results_comparison.png')))

## Cell 10 — Save results and weights to Google Drive

In [ ]:
import shutil, os

DRIVE_OUTPUT  = '/content/drive/MyDrive/NEUSTA/Results'
DRIVE_WEIGHTS = os.path.join(DRIVE_OUTPUT, 'weights')
os.makedirs(DRIVE_OUTPUT,  exist_ok=True)
os.makedirs(DRIVE_WEIGHTS, exist_ok=True)

# Results (CSV, PNG, JSON)
for fname in os.listdir(RESULTS_DIR):
    src = os.path.join(RESULTS_DIR, fname)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join(DRIVE_OUTPUT, fname))
        print(f'Copied: {fname}')

# Weights
shutil.copy(RGB_WEIGHTS,  os.path.join(DRIVE_WEIGHTS, 'best_rgb.pt'))
shutil.copy(RGBN_WEIGHTS, os.path.join(DRIVE_WEIGHTS, 'best_rgbn.pt'))
print('Copied: best_rgb.pt')
print('Copied: best_rgbn.pt')
print(f'\nAll saved to: {DRIVE_OUTPUT}')

## Cell 11 — Show sample prediction visualisations

In [ ]:
import os, matplotlib.pyplot as plt, matplotlib.image as mpimg

viz_dir   = os.path.join(RESULTS_DIR, 'viz')
viz_files = sorted([f for f in os.listdir(viz_dir) if f.startswith('rgb_')])[:4]

fig, axes = plt.subplots(1, len(viz_files), figsize=(16, 4))
fig.suptitle('RGB Predictions — green=ground truth, red=predicted', fontsize=12)
for ax, fname in zip(axes, viz_files):
    ax.imshow(mpimg.imread(os.path.join(viz_dir, fname)))
    ax.set_title(fname.replace('rgb_', ''), fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'sample_predictions_rgb.png'), dpi=150)
plt.show()
print('Saved to Drive.')